El objetivo de este Notebook es identificar conexiones (whois, IP, reverse IP)entre los dominios obtenidos en el Notebook 06.

In [15]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
cache  data  models  notebooks	README.md  requirements.txt  results


In [12]:
import pandas as pd
import os
import json

import time
import socket

from collections import defaultdict

import subprocess


# Rutas del proyecto

In [8]:
DATA_PATH = "data/fnn_processed.csv"
CACHE_DIR = "cache"    # Defino el directorio donde se irá guardando el caché
RESULTS_DIR = "results"

# Nota: dominio -> IP -> ASN

WHOIS_CACHE = os.path.join(CACHE_DIR, "whois")
IP_CACHE = os.path.join(CACHE_DIR, "ip")    # Aquí guardo las IP de los dominios
ASN_CACHE = os.path.join(CACHE_DIR, "asn")   # Aquí guardaré el ASN (quien controla la red)

# Creo las carpetas en el caso en el que no existan
for path in [CACHE_DIR, WHOIS_CACHE, IP_CACHE, ASN_CACHE, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)


In [9]:
# Cargo el dataset base del grafo
df_fnn = pd.read_csv(DATA_PATH)

print("Noticias cargadas:", len(df_fnn))
print("Dominios únicos:", df_fnn["domain_norm"].nunique())

df_fnn.head()


Noticias cargadas: 22866
Dominios únicos: 2429


,id,news_url,domain_norm,label,source_dataset
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,speedtalk.com,1,politifact
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,politics2020.info,1,politifact
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,nscdscamps.org,1,politifact
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,howafrica.com,1,politifact
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,washingtonsources.org,1,politifact


In [11]:
# Extracción de los 10 primeros dominios únicos (los nodos del grafo)
domains = sorted(df_fnn["domain_norm"].unique())

print("Total de dominios:", len(domains))
domains[:10]


Total de dominios: 2429


['1001.com.do',
 '100percentfedup.com',
 '101kgb.iheart.com',
 '1029now.iheart.com',
 '1037theq.iheart.com',
 '1043myfm.iheart.com',
 '1061kissfm.iheart.com',
 '10best.com',
 '10news.com',
 '12news.com']

# Resolución IP a partir del dominio

In [13]:
def get_ip(domain):
    cache_file = os.path.join(IP_CACHE, f"{domain}.json")

    if os.path.exists(cache_file):     # Compruebo si ya existe el archivo
        with open(cache_file, "r") as f:
            return json.load(f)

    try:
        ip = socket.gethostbyname(domain)      # Pregunta a DNS cual es la IP del dominio
        result = {"domain": domain, "ip": ip}
    except Exception:
        result = {"domain": domain, "ip": None}

    with open(cache_file, "w") as f:
        json.dump(result, f)    # Guardado de resultados en el caché

    time.sleep(0.1)   # Pausa, para reducir bloqueos

    return result


# WHOIS básico

In [14]:
def get_whois(domain):
    cache_file = os.path.join(WHOIS_CACHE, f"{domain}.json")

    if os.path.exists(cache_file):     # Compruebo si ya existe el archivo
        with open(cache_file, "r") as f:
            return json.load(f)

    result = {"domain": domain, "registrar": None}    # Garantizo que la función tenga siempre el mismo formato

    try:
        output = subprocess.check_output(
            ["whois", domain],
            stderr= subprocess.DEVNULL,   # Ignora mensajes de error del comando
            timeout= 10    # Si tarda mas de 10 seg, aborta
        ).decode(errors="ignore")

        for line in output.splitlines():
            if "Registrar:" in line:
                result["registrar"] = line.split(":", 1)[1].strip()   # Me quedo con el resultado de la línea de Registrar
                break
    except Exception:
        pass

    with open(cache_file, "w") as f:
        json.dump(result, f)    # Guardado de resultados en el caché

    time.sleep(0.2)   # Pausa, para reducir bloqueos (más larga que la de IP)

    return result


# Infraestructura por dominio

In [17]:
infra = {}

for domain in domains:
    ip_info = get_ip(domain)
    whois_info = get_whois(domain)

    infra[domain] = {
        "ip": ip_info.get("ip"),
        "registrar": whois_info.get("registrar"),
    }


# Generación de aristas

In [18]:
edges = []    # Aquí se guardarán todas las aristas del grafo

registrar_groups = defaultdict(list)    # Quiero agrupar dominios que tengan el mismo "registrar"
for d, info in infra.items():
    if info["registrar"]:
        registrar_groups[info["registrar"]].append(d)   # Añade el dominio a la lista asociada a registrar

for registrar, doms in registrar_groups.items():   # Recorre cada grupo
    if len(doms) > 1:    # Comprueba que hay 2 dominios por grupo
        for i in range(len(doms)):
            for j in range(i + 1, len(doms)):    # Generación de combinaciones de dominios (sin repetirse)
                edges.append({    # Añade nueva arista
                    "source": doms[i],
                    "target": doms[j],
                    "edge_type": "same_registrar",
                    "evidence": registrar
                })


ip_groups = defaultdict(list)     # Repito lo mismo pero agrupando dominios con la misma IP
for d, info in infra.items():
    if info["ip"]:
        ip_groups[info["ip"]].append(d)

for ip, doms in ip_groups.items():
    if len(doms) > 1:
        for i in range(len(doms)):
            for j in range(i + 1, len(doms)):
                edges.append({
                    "source": doms[i],
                    "target": doms[j],
                    "edge_type": "same_ip",
                    "evidence": ip
                })


In [19]:
# Creación DataFrame final de aristas
df_edges = pd.DataFrame(edges)

print("Aristas generadas:", len(df_edges))
df_edges.head()


Aristas generadas: 3874


,source,target,edge_type,evidence
0,101kgb.iheart.com,1029now.iheart.com,same_ip,199.232.210.193
1,101kgb.iheart.com,1037theq.iheart.com,same_ip,199.232.210.193
2,101kgb.iheart.com,1043myfm.iheart.com,same_ip,199.232.210.193
3,101kgb.iheart.com,1061kissfm.iheart.com,same_ip,199.232.210.193
4,101kgb.iheart.com,939litefm.iheart.com,same_ip,199.232.210.193


# Guardado de los resultados

In [20]:
OUTPUT_PATH = os.path.join(RESULTS_DIR, "whois_edges.csv")
df_edges.to_csv(OUTPUT_PATH, index=False)

print(f"Aristas guardadas en {OUTPUT_PATH}")


Aristas guardadas en results/whois_edges.csv
